# 03 — Text Extraction and Cleaning

Extract one record per physical PDF page and apply light, auditable removal of repeated headers,
footers, and page-number lines. Raw text is retained for traceability.

In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "intelligent_scada").is_dir():
            return candidate
    raise RuntimeError("Could not locate the Intelligent SCADA project root.")

project_root = find_project_root(Path.cwd().resolve())
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from intelligent_scada.config import Settings

settings = Settings.from_env(project_root=project_root, env_override=True).for_default_document()
settings.ensure_output_dirs()


In [ ]:
from intelligent_scada.ingestion.text_extractor import extract_page_text
from intelligent_scada.ingestion.text_cleaner import clean_pages

raw_pages = extract_page_text(settings.require_default_pdf(), settings.raw_text_path)
cleaned_pages = clean_pages(
    raw_pages,
    settings.clean_text_path,
    header_footer_patterns=settings.header_footer_patterns,
    page_number_pattern=settings.page_number_pattern,
)

removed_lines = sum(len(page["removed_lines"]) for page in cleaned_pages)
print(f"Pages: {len(cleaned_pages)}")
print(f"Pages with text: {sum(page['has_text'] for page in cleaned_pages)}")
print(f"Removed lines: {removed_lines}")
print(f"Output: {settings.clean_text_path}")
